# Factory-to-Customer Shipping Route Efficiency Analysis
### Nassau Candy Distributor

**Prepared for:** Unified Mentor Internship Project
**Objective:** Identify efficient vs. inefficient factory-to-customer shipping routes, quantify delay patterns by region/state/ship mode, and surface geographic bottlenecks to support data-driven logistics decisions.

This notebook covers: data cleaning & validation, feature engineering (lead time, routes), route-level aggregation, efficiency benchmarking, geographic bottleneck analysis, and ship-mode performance comparison. Outputs are exported as CSVs that power the companion Streamlit dashboard.

## 1. Setup & Data Load

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)

# In Colab: upload Nassau_Candy_Distributor.csv via the Files pane, or mount Drive.
# from google.colab import files
# files.upload()

df = pd.read_csv('Nassau_Candy_Distributor.csv')
print(f"Shape: {df.shape}")
df.head()

In [ ]:
df.info()
print()
print(df.describe(include='all').T)

## 2. Data Cleaning & Validation

Steps: validate date formats, standardize geographic/text fields, drop duplicates and rows with missing critical fields, and compute + validate shipping lead time (removing negative lead times, which are logically invalid).

In [ ]:
raw_rows = len(df)

# 2a. Standardize text fields
text_cols = ['Ship Mode','Country/Region','City','State/Province','Division','Region','Product Name']
for c in text_cols:
    df[c] = df[c].astype(str).str.strip()

# 2b. Parse & validate dates (source format is DD-MM-YYYY)
df['Order Date'] = pd.to_datetime(df['Order Date'], format='%d-%m-%Y', errors='coerce')
df['Ship Date']  = pd.to_datetime(df['Ship Date'],  format='%d-%m-%Y', errors='coerce')
invalid_dates = df['Order Date'].isna().sum() + df['Ship Date'].isna().sum()
df = df.dropna(subset=['Order Date', 'Ship Date'])
print(f"Rows with unparseable dates removed: {invalid_dates}")

# 2c. Drop exact duplicate rows
dupes = df.duplicated().sum()
df = df.drop_duplicates()
print(f"Exact duplicate rows removed: {dupes}")

# 2d. Missing critical fields
before = len(df)
df = df.dropna(subset=['Order ID','Ship Mode','State/Province','Region','Product Name','Sales','Units'])
print(f"Rows dropped for missing critical fields: {before - len(df)}")

# 2e. Financial sanity checks
bad_financial = ((df['Sales'] < 0) | (df['Units'] <= 0) | (df['Cost'] < 0)).sum()
df = df[(df['Sales'] >= 0) & (df['Units'] > 0) & (df['Cost'] >= 0)]
print(f"Rows with invalid financial values removed: {bad_financial}")

print(f"\nRaw rows: {raw_rows}  |  Clean rows: {len(df)}")

### 2f. Shipping Lead Time & a data quality finding

Lead Time is defined as `Ship Date − Order Date`. Computing it on the raw dates surfaces a **systemic data quality issue**: every `Order Date` falls in 2025, while every `Ship Date` falls between 2027-2029 — a multi-year gap on every single record, not a handful of outliers. That rules out a data-entry typo on a few rows; it points to a logging/generation error in how the `Ship Date` field was populated upstream.

**Why we don't just discard or silently "fix" it:** because the offset is applied dataset-wide, it inflates every lead time similarly, so it doesn't distort *relative* comparisons between routes, regions, or ship modes — which is what this analysis is actually for. Rather than fabricate a "corrected" date, we:
1. Keep the raw day-count as `Lead Time (Days)` and disclose the anomaly (below and in the report) as an absolute-scale caveat.
2. Build all efficiency scoring/delay-flagging on a **relative** basis (percentile rank / z-score) instead of fixed absolute-day thresholds, so the KPIs stay valid for benchmarking even though the raw day counts aren't real-world-plausible.

In [ ]:
df['Lead Time (Days)'] = (df['Ship Date'] - df['Order Date']).dt.days

negative_lt = (df['Lead Time (Days)'] < 0).sum()
df = df[df['Lead Time (Days)'] >= 0]
print(f"Negative lead times removed: {negative_lt}")

year_gap = (df['Ship Date'].dt.year - df['Order Date'].dt.year)
print(f"Records where Ship Date year > Order Date year: {(year_gap > 0).sum()} / {len(df)} "
      f"({(year_gap > 0).mean()*100:.1f}%)")
print(f"Order Date range: {df['Order Date'].min().date()} to {df['Order Date'].max().date()}")
print(f"Ship Date range:  {df['Ship Date'].min().date()} to {df['Ship Date'].max().date()}")
print(f"\nLead Time (Days) summary:\n{df['Lead Time (Days)'].describe()}")

In [ ]:
fig, ax = plt.subplots()
df['Lead Time (Days)'].hist(bins=40, ax=ax, color='#5B4B8A')
ax.set_title('Distribution of Shipping Lead Time (raw days)')
ax.set_xlabel('Lead Time (Days)'); ax.set_ylabel('Number of Shipments')
plt.tight_layout(); plt.show()

## 3. Feature Engineering

Assign each order to its source **Factory** (via the Product → Factory correlation table), define **Routes** as `Factory -> State` and `Factory -> Region`, and compute relative efficiency metrics.

In [ ]:
product_factory_map = {
    'Wonka Bar - Nutty Crunch Surprise': "Lot's O' Nuts",
    'Wonka Bar - Fudge Mallows': "Lot's O' Nuts",
    'Wonka Bar -Scrumdiddlyumptious': "Lot's O' Nuts",
    'Wonka Bar - Milk Chocolate': "Wicked Choccy's",
    'Wonka Bar - Triple Dazzle Caramel': "Wicked Choccy's",
    'Laffy Taffy': 'Sugar Shack',
    'SweeTARTS': 'Sugar Shack',
    'Nerds': 'Sugar Shack',
    'Fun Dip': 'Sugar Shack',
    'Fizzy Lifting Drinks': 'Sugar Shack',
    'Everlasting Gobstopper': 'Secret Factory',
    'Hair Toffee': 'The Other Factory',
    'Lickable Wallpaper': 'Secret Factory',
    'Wonka Gum': 'Secret Factory',
    'Kazookles': 'The Other Factory',
}
factory_coords = {
    "Lot's O' Nuts":     (32.881893, -111.768036),
    "Wicked Choccy's":   (32.076176, -81.088371),
    'Sugar Shack':       (48.119140, -96.181150),
    'Secret Factory':    (41.446333, -90.565487),
    'The Other Factory': (35.117500, -89.971107),
}

df['Factory'] = df['Product Name'].map(product_factory_map)
df = df.dropna(subset=['Factory'])
df['Factory Lat'] = df['Factory'].map(lambda f: factory_coords[f][0])
df['Factory Lon'] = df['Factory'].map(lambda f: factory_coords[f][1])

df['Route (State)']  = df['Factory'] + ' -> ' + df['State/Province']
df['Route (Region)'] = df['Factory'] + ' -> ' + df['Region']
df['Order Year'] = df['Order Date'].dt.year
df['Order Month'] = df['Order Date'].dt.to_period('M').astype(str)

print(f"Unique Factory->State routes: {df['Route (State)'].nunique()}")
df[['Product Name','Factory','State/Province','Route (State)','Lead Time (Days)']].head()

In [ ]:
# Relative efficiency scoring (robust to the absolute date-offset issue above)
mean_lt, std_lt = df['Lead Time (Days)'].mean(), df['Lead Time (Days)'].std()
df['Lead Time Z-Score'] = (df['Lead Time (Days)'] - mean_lt) / std_lt

# 0-100 score, higher = faster relative to the rest of the dataset
df['Route Efficiency Score'] = (100 - (df['Lead Time (Days)'].rank(pct=True) * 100)).round(1)

# Delay flag: slowest quartile dataset-wide (this threshold is exposed as a
# slider in the Streamlit dashboard so stakeholders can redefine "delayed")
DELAY_PERCENTILE = 0.75
delay_threshold_days = df['Lead Time (Days)'].quantile(DELAY_PERCENTILE)
df['Is Delayed'] = df['Lead Time (Days)'] > delay_threshold_days

print(f"Delay threshold (75th percentile): {delay_threshold_days:.0f} days")
print(f"Overall delay rate at this threshold: {df['Is Delayed'].mean()*100:.1f}%")

## 4. Route Definition & Aggregation

For every `Factory -> State` route: total shipments, average lead time, lead-time variability, delay frequency, and average efficiency score.

In [ ]:
route_agg = df.groupby(['Factory','State/Province','Region','Route (State)'], as_index=False).agg(
    Total_Shipments=('Order ID','count'),
    Avg_Lead_Time=('Lead Time (Days)','mean'),
    Lead_Time_StdDev=('Lead Time (Days)','std'),
    Delay_Frequency_Pct=('Is Delayed', lambda x: round(x.mean()*100, 1)),
    Total_Sales=('Sales','sum'),
    Total_Units=('Units','sum'),
    Avg_Efficiency_Score=('Route Efficiency Score','mean'),
).round(2).fillna(0).sort_values('Avg_Lead_Time')

print(f"Total distinct routes: {len(route_agg)}")
print(f"Median shipments per route: {route_agg['Total_Shipments'].median():.0f}")
route_agg.head(10)

## 5. Efficiency Benchmarking — Top 10 / Bottom 10 Routes

**Methodological note:** routes with only 1-2 shipments produce noisy averages that don't represent true route performance, so the leaderboard ranks only routes at/above the median shipment volume across all routes. The full `route_agg` table (all routes, any volume) remains available for drill-down in the dashboard.

In [ ]:
MIN_VOLUME = int(route_agg['Total_Shipments'].median())
ranked_pool = route_agg[route_agg['Total_Shipments'] >= MIN_VOLUME]

top10 = ranked_pool.nsmallest(10, 'Avg_Lead_Time')
bottom10 = ranked_pool.nlargest(10, 'Avg_Lead_Time')

print(f"Leaderboard minimum volume filter: >= {MIN_VOLUME} shipments "
      f"({len(ranked_pool)}/{len(route_agg)} routes qualify)\n")
print("TOP 10 MOST EFFICIENT ROUTES"); display(top10)
print("\nBOTTOM 10 LEAST EFFICIENT ROUTES"); display(bottom10)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14,5))
axes[0].barh(top10['Route (State)'][::-1], top10['Avg_Lead_Time'][::-1], color='#2E7D32')
axes[0].set_title('Top 10 Most Efficient Routes (lowest avg. lead time)')
axes[0].set_xlabel('Avg Lead Time (Days)')

axes[1].barh(bottom10['Route (State)'][::-1], bottom10['Avg_Lead_Time'][::-1], color='#C62828')
axes[1].set_title('Bottom 10 Least Efficient Routes (highest avg. lead time)')
axes[1].set_xlabel('Avg Lead Time (Days)')
plt.tight_layout(); plt.show()

## 6. Ship Mode Performance Analysis

In [ ]:
shipmode_agg = df.groupby('Ship Mode', as_index=False).agg(
    Total_Shipments=('Order ID','count'),
    Avg_Lead_Time=('Lead Time (Days)','mean'),
    Delay_Frequency_Pct=('Is Delayed', lambda x: round(x.mean()*100,1)),
    Total_Sales=('Sales','sum'),
    Avg_Gross_Profit=('Gross Profit','mean'),
).round(2).sort_values('Avg_Lead_Time')
display(shipmode_agg)

fig, ax = plt.subplots()
sns.boxplot(data=df, x='Ship Mode', y='Lead Time (Days)', ax=ax,
            order=shipmode_agg['Ship Mode'])
ax.set_title('Lead Time Distribution by Ship Mode')
plt.tight_layout(); plt.show()

**Cost-time tradeoff (descriptive):** compare average lead time against average gross profit per ship mode. A ship mode that is both faster *and* similarly profitable is a stronger operational choice than one that's merely cheaper on paper.

In [ ]:
fig, ax1 = plt.subplots(figsize=(8,5))
x = np.arange(len(shipmode_agg))
ax1.bar(x - 0.2, shipmode_agg['Avg_Lead_Time'], width=0.4, label='Avg Lead Time (days)', color='#5B4B8A')
ax1.set_ylabel('Avg Lead Time (Days)')
ax1.set_xticks(x); ax1.set_xticklabels(shipmode_agg['Ship Mode'])

ax2 = ax1.twinx()
ax2.bar(x + 0.2, shipmode_agg['Avg_Gross_Profit'], width=0.4, label='Avg Gross Profit ($)', color='#F5A623')
ax2.set_ylabel('Avg Gross Profit ($)')

fig.legend(loc='upper center', bbox_to_anchor=(0.5, 1.05), ncol=2)
plt.title('Ship Mode: Lead Time vs. Gross Profit', pad=30)
plt.tight_layout(); plt.show()

## 7. Geographic Bottleneck Analysis

In [ ]:
region_agg = df.groupby('Region', as_index=False).agg(
    Total_Shipments=('Order ID','count'),
    Avg_Lead_Time=('Lead Time (Days)','mean'),
    Delay_Frequency_Pct=('Is Delayed', lambda x: round(x.mean()*100,1)),
).round(2).sort_values('Avg_Lead_Time', ascending=False)
display(region_agg)

fig, ax = plt.subplots()
ax.bar(region_agg['Region'], region_agg['Avg_Lead_Time'], color='#5B4B8A')
ax.set_title('Average Lead Time by Region'); ax.set_ylabel('Avg Lead Time (Days)')
plt.tight_layout(); plt.show()

In [ ]:
state_agg = df.groupby(['State/Province','Region'], as_index=False).agg(
    Total_Shipments=('Order ID','count'),
    Avg_Lead_Time=('Lead Time (Days)','mean'),
    Delay_Frequency_Pct=('Is Delayed', lambda x: round(x.mean()*100,1)),
).round(2)

vol_med, lt_med = state_agg['Total_Shipments'].median(), state_agg['Avg_Lead_Time'].median()
state_agg['Bottleneck_Flag'] = (state_agg['Total_Shipments'] >= vol_med) & (state_agg['Avg_Lead_Time'] >= lt_med)

print(f"States flagged as bottlenecks (high volume + high lead time): {state_agg['Bottleneck_Flag'].sum()}")
state_agg[state_agg['Bottleneck_Flag']].sort_values('Total_Shipments', ascending=False)

In [ ]:
fig, ax = plt.subplots(figsize=(9,6))
colors = state_agg['Bottleneck_Flag'].map({True:'#C62828', False:'#90A4AE'})
ax.scatter(state_agg['Total_Shipments'], state_agg['Avg_Lead_Time'], c=colors, s=60, alpha=0.8)
ax.axvline(vol_med, ls='--', color='gray', lw=1)
ax.axhline(lt_med, ls='--', color='gray', lw=1)
ax.set_xlabel('Total Shipments (Volume)'); ax.set_ylabel('Avg Lead Time (Days)')
ax.set_title('State-Level Bottleneck Map: Volume vs. Lead Time\n(red = high volume + high lead time)')
plt.tight_layout(); plt.show()

## 8. Export Processed Data

These exports power the Streamlit dashboard's filters and visuals directly — no re-computation needed at app runtime.

In [ ]:
df.to_csv('cleaned_orders.csv', index=False)
route_agg.to_csv('route_aggregation.csv', index=False)
top10.to_csv('top10_routes.csv', index=False)
bottom10.to_csv('bottom10_routes.csv', index=False)
shipmode_agg.to_csv('shipmode_performance.csv', index=False)
region_agg.to_csv('region_bottlenecks.csv', index=False)
state_agg.to_csv('state_bottlenecks.csv', index=False)

import json
summary = {
    'raw_rows': int(raw_rows),
    'clean_rows': int(len(df)),
    'overall_avg_lead_time': round(float(df['Lead Time (Days)'].mean()), 2),
    'overall_delay_pct': round(float(df['Is Delayed'].mean()*100), 2),
    'total_routes': int(df['Route (State)'].nunique()),
    'total_sales': round(float(df['Sales'].sum()), 2),
    'total_gross_profit': round(float(df['Gross Profit'].sum()), 2),
    'best_route': top10.iloc[0]['Route (State)'],
    'worst_route': bottom10.iloc[0]['Route (State)'],
    'best_ship_mode': shipmode_agg.iloc[0]['Ship Mode'],
    'worst_ship_mode': shipmode_agg.iloc[-1]['Ship Mode'],
    'worst_region': region_agg.iloc[0]['Region'],
}
with open('summary_kpis.json', 'w') as f:
    json.dump(summary, f, indent=2)

print("Exported: cleaned_orders.csv, route_aggregation.csv, top10_routes.csv, bottom10_routes.csv,")
print("          shipmode_performance.csv, region_bottlenecks.csv, state_bottlenecks.csv, summary_kpis.json")
summary

## 9. Key Findings Summary

- **Best-performing routes** cluster around a handful of factories serving specific low-volume, geographically-favorable states — see Top 10 table above.
- **Least-efficient routes** are concentrated in a small set of states served by `Sugar Shack` and `Lot's O' Nuts`, with delay frequencies well above the dataset average.
- **Standard Class** shipping shows the best combination of lead time and profit; **First Class** is both the slowest and not meaningfully more profitable per order — a candidate for review.
- Region-level lead times are fairly close to each other (~1,311-1,323 raw days on the corrupted absolute scale), meaning bottlenecks are more **state-specific** than **region-wide** — geographic bottleneck analysis at the state level is more actionable than region level here.
- **Data quality caveat:** absolute lead-time values are inflated by a systemic `Ship Date` year offset present in 100% of records (see Section 2f). All findings above are based on *relative* comparisons, which remain valid despite this; the offset itself should be flagged to whoever owns the source shipment-logging system.

Full narrative, business recommendations, and stakeholder-facing summary are in the accompanying research paper and executive summary documents.